In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv(r'/kaggle/input/datasets/adhurimquku/ford-car-price-prediction/ford.csv')

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

# EDA

In [ ]:
cols = ['year','price','mileage','tax','mpg','engineSize']


plt.figure(figsize=(13,12))

for i,col in enumerate(cols):
    plt.subplot(3,2,1+i)
    sns.histplot(df[col], bins=30, kde=True)
    plt.title(f"Histplot of {col}")

plt.tight_layout()
plt.show()

In [ ]:
cols = ['year','price','mileage','tax','mpg','engineSize']


plt.figure(figsize=(13,10))

for i,col in enumerate(cols):
    plt.subplot(3,2,1+i)
    sns.boxplot(df[col])
    plt.title(f"Boxplot of {col}")

plt.tight_layout()
plt.show()

In [ ]:
cols = ['year','mileage','tax','mpg','engineSize']


plt.figure(figsize=(13,12))

for i,col in enumerate(cols):
    plt.subplot(3,2,1+i)
    sns.scatterplot(data=df, x=col, y=df['price'])
    plt.title(f"Relations between {col} and price")

plt.tight_layout()
plt.show()

In [ ]:
df.columns

In [ ]:
cols = ['year','engineSize','model','transmission','fuelType']


plt.figure(figsize=(13,12))

for i,col in enumerate(cols):
    plt.subplot(3,2,1+i)
    sns.boxplot(data= df, x = col, y=df['price'], palette='rainbow')
    plt.title(f"Relations between {col} and price")
    plt.xticks(rotation=90)

plt.tight_layout()
plt.show()

In [ ]:
corr = df.select_dtypes(include='number').corr()
sns.heatmap(corr, annot=True)
plt.show()

# Data Cleaning and Preprocessing

In [ ]:
print(f'Before deleting duplicates: {df.shape[0]}')

df.drop_duplicates(inplace=True)

print(f'After deleting duplicates: {df.shape[0]}')

In [ ]:
mode_year = df['year'].mode()[0]

df['year'] = df['year'].replace(2060, mode_year)

In [ ]:
X = df.drop('price', axis=1)
y = df['price']

**One-Hot-Encoding**

In [ ]:
X_ohe = pd.get_dummies(
    data = X,
    columns=['model','transmission','fuelType'],
    drop_first= True
).astype(int)


**Label-Encoding**

In [ ]:
encoder = LabelEncoder()

cols = ['model','transmission','fuelType']
X_label = X.copy()

for col in cols:
    X_label[col] = encoder.fit_transform(df[col])

**Scaling One-Hot-Encoding & Label-Encoding Data**

In [ ]:
scaler = StandardScaler()

cols = ['year','mileage','tax','mpg','engineSize']
X_ohe_ss = X_ohe.copy()
X_label_ss = X_label.copy()

X_ohe_ss[cols] = scaler.fit_transform(X_ohe_ss[cols])

X_label_ss = pd.DataFrame(
    data = scaler.fit_transform(X_label_ss),
    columns=X_label_ss.columns
)

# Feature Selection

**Features Importances In Label-Encodig Data**

In [ ]:
model = RandomForestRegressor(random_state=42)
model.fit(X_label, y)

importance = model.feature_importances_.round(2)*100

for col,imp in zip(X_label.columns, importance):
    print(f"{col}:{imp}%")

**Features Importances In One-Hot-Encodig Data**

In [ ]:
model = RandomForestRegressor(random_state=42)
model.fit(X_ohe, y)

importance = model.feature_importances_.round(2)*100

for col,imp in zip(X_ohe.columns, importance):
    print(f"{col} : {imp}%")

# Cross Validation

 **In One-Hot-Encoding Data**

In [ ]:
model = LinearRegression()
model_2 = SVR()
model_3 = DecisionTreeRegressor()
model_4 = RandomForestRegressor()
model_5 = XGBRegressor()

models = {
    'lr':model,
    'svr':model_2,
    'dt':model_3,
    'rf':model_4,
    'xgb':model_5
}

for name,model in models.items():
    scores = cross_val_score(model, X_ohe, y, cv=5)
    
    print(f'----------{name}---------')
    
    print(f'All Score: {scores}')
    print(f'Average Score: {scores.mean()}')
    print('-'*30)

**In Label-Data**

In [ ]:
model = LinearRegression()
model_2 = SVR()
model_3 = DecisionTreeRegressor()
model_4 = RandomForestRegressor()
model_5 = XGBRegressor()

models = {
    'lr':model,
    'svr':model_2,
    'dt':model_3,
    'rf':model_4,
    'xgb':model_5
}

for name,model in models.items():
    scores = cross_val_score(model, X_label, y, cv=5)
    
    print(f'----------{name}---------')
    
    print(f'All Score: {scores}')
    print(f'Average Score: {scores.mean()}')
    print('-'*30)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_ohe, y, test_size=0.2, random_state=42)

In [ ]:
model = XGBRegressor(
    subsample= 0.6,
    n_estimators= 60,
    max_leaves= 14,
    max_depth= 19,
    learning_rate= 0.2,
    colsample_bytree= 1.0
)

model.fit(X_train, y_train)

model.score(X_train, y_train),model.score(X_test, y_test)

In [ ]:
params = {
    'max_depth': [n for n in range(1, 20)],
    'max_leaves': [n for n in range(1, 40)],
    'n_estimators': [n for n in range(50, 200, 5)],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0]
}

In [ ]:
random_search = RandomizedSearchCV(
    model,
    param_distributions= params,
    cv = 5,
    scoring= 'accuracy',
    random_state= 42
)

random_search.fit(X_train, y_train)

print('Best Params:', random_search.best_params_)
print('Best Score:', random_search.best_score_)

# Model Evaluation

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
r2 = r2_score(y_test, y_pred)
n = X_test.shape[0]
p = X_test.shape[1]

adjusted_r2 = 1 - ((1-r2)*(n-1)/(n-p-1))

print(f"R2:", r2)
print(f"Adjusted R2: {adjusted_r2}")
print(f'MAE: {mean_absolute_error(y_test, y_pred)}')
print(f'MSE: {mean_squared_error(y_test, y_pred)}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, y_pred))}')


In [ ]:
import joblib

joblib.dump(model, 'Ford_Car_Price_Prediction.pkl')